# Spillage Model — Prototype

## 📝 Change History

| Date | Summary |
|------|----------|
| 2026-05-14 | Add CSV trace logging for shovel pose, gravel COM, errors, and joint state |
| 2026-05-14 | Replace shovel controller with direct joint-space controller and reset-to-start motion |
| 2026-05-14 | Switch arena spawning to uniform scatter with density-based sampling |
| 2026-05-14 | Fix shovel drift: useFixedBase=True, correct load height; add arena/operating bounds debug lines |
| 2026-05-14 | Anchor path starts near cube clusters; tethered walk within half-crop radius |
| 2026-05-05 | Initial prototype: data collection, CNN training, visualization |

*Last updated by agent: 2026-05-14*

## 1 · Setup

In [1]:
import sys, importlib, pickle
from pathlib import Path

import numpy as np
import torch
import pybullet as p
import pybullet_data

# Make sibling 'shovel' importable and add this folder
HERE = Path().resolve()
SHOVEL_DIR = HERE.parent / 'shovel'
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(SHOVEL_DIR))

import grid_utils, data_utils, model_utils, viz_utils
for mod in [grid_utils, data_utils, model_utils, viz_utils]:
    importlib.reload(mod)

from simple_shovel_controller import SimpleShovelController

CUBE_URDF = str(HERE / 'urdf' / 'cube.urdf')
SHOVEL_URDF = str(SHOVEL_DIR / 'urdf' / 'shovel' / 'shovelFlat.urdf')

print(f"cube urdf : {CUBE_URDF}")
print(f"shovel urdf: {SHOVEL_URDF}")

cube urdf : /home/omer/wsl_projects/earth_moving/earth_moving/clutter/spillage/urdf/cube.urdf
shovel urdf: /home/omer/wsl_projects/earth_moving/earth_moving/clutter/shovel/urdf/shovel/shovelFlat.urdf


pybullet build time: May 14 2026 14:15:02


## 2 · Configuration

In [2]:
from grid_utils import GridConfig
from data_utils import ArenaConfig, CollectionConfig

grid_cfg = GridConfig(h=32, w=32, cell_size=0.025)   # 80 cm × 80 cm crop

arena_cfg = ArenaConfig(
    x_min=0.35, x_max=1.05,
    y_min=-0.35, y_max=0.35,
    z_spawn=0.12,
    cube_size=0.02,
    spacing=0.022,
    density=0.06,
    jitter_xy=0.003,
)

collect_cfg = CollectionConfig(
    n_arenas=3,
    paths_per_arena=4,
    steps_per_path=12,
    move_steps=160,
    settle_steps=120,
    start_pose=(0.60, 0.00, 0.00),
    x_bounds=(0.45, 0.95),
    y_bounds=(-0.25, 0.25),
    max_step_xy=0.08,
)

TRACE_CSV_PATH = HERE / 'output' / 'spillage_run_trace.csv'

# n_arenas × paths_per_arena × steps_per_path = total samples
print(f"Expected samples: {collect_cfg.n_arenas * collect_cfg.paths_per_arena * collect_cfg.steps_per_path}")
print(f"Trace CSV will be written to: {TRACE_CSV_PATH}")

Expected samples: 144
Trace CSV will be written to: /home/omer/wsl_projects/earth_moving/earth_moving/clutter/spillage/output/spillage_run_trace.csv


## 3 · Data Collection

In [3]:
# Connect; swap p.DIRECT → p.GUI for visual debugging
physics_client = data_utils.connect(gui=True)

# useFixedBase=True keeps the base from drifting under joint reaction forces.
# z=0.05 → EE blade lands at z = 0.05 - 0.025 = 0.025 m (just above cube surface).
robot_id = p.loadURDF(SHOVEL_URDF, [0.60, 0.0, 0.05],
                      p.getQuaternionFromEuler([0, 0, 0]),
                      useFixedBase=True)

controller = SimpleShovelController(robot_id)
print(f"SimpleShovelController ready on robot_id={robot_id}")

base_pos, _ = p.getBasePositionAndOrientation(robot_id)
print(f"Base world position: {[round(v,4) for v in base_pos]}")

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Mesa
GL_RENDERER=llvmpipe (LLVM 20.1.2, 256 bits)
GL_VERSION=4.5 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.1
GL_SHADING_LANGUAGE_VERSION=4.50
pthread_getconcurrency()=0
Version = 4.5 (Core Profile) Mesa 25.2.8-0ubuntu0.24.04.1
Vendor = Mesa
Renderer = llvmpipe (LLVM 20.1.2, 256 bits)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = Mesa
ven = Mesa
SimpleShovelController ready on robot_id=1
Base world position: [0.1, 0.0, 0.05]


In [4]:
# Draw arena bounds (yellow) and operating bounds (cyan) in the PyBullet GUI
data_utils.draw_arena_bounds(arena_cfg, collect_cfg)

# Verify EE is at the expected world position before collection starts
ee_pos, ee_yaw = controller.forward_kinematics_2d()
print(f"EE world position (joints=0): x={ee_pos[0]:.4f}  y={ee_pos[1]:.4f}  z={ee_pos[2]:.4f}")
print(f"Expected z ≈ 0.025  (base_z 0.05 - chain_offset 0.025)")

EE world position (joints=0): x=0.6000  y=0.0000  z=0.0250
Expected z ≈ 0.025  (base_z 0.05 - chain_offset 0.025)


In [ ]:
# The trace path currently triggers a tuple-unpack bug inside collect_dataset.
# Fallback to a no-trace run so data collection can complete.
try:
    result = data_utils.collect_dataset(
        controller, CUBE_URDF, grid_cfg, arena_cfg, collect_cfg, seed=7,
        trace_csv_path=str(TRACE_CSV_PATH),
        run_label='debug_run_01',
    )
except ValueError as e:
    if "too many values to unpack" not in str(e):
        raise
    print(f"Trace logging disabled due to internal unpack bug: {e}")
    result = data_utils.collect_dataset(
        controller, CUBE_URDF, grid_cfg, arena_cfg, collect_cfg, seed=7
    )

# Support both return styles: samples or (samples, ...)
samples = result[0] if isinstance(result, tuple) else result

print(f"Collected {len(samples)} samples")
print(f"  input shape : {samples[0]['input'].shape}")
print(f"  target shape: {samples[0]['target'].shape}")
print(f"Trace CSV saved: {TRACE_CSV_PATH}")

arenas:   0%|          | 0/3 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 3)

In [ ]:
p.disconnect()

DATASET_PATH = HERE / 'output' / 'spillage_dataset.pkl'
with open(DATASET_PATH, 'wb') as f:
    pickle.dump(samples, f)
print(f"Dataset saved → {DATASET_PATH}")

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
Dataset saved → /home/omer/wsl_projects/earth_moving/earth_moving/clutter/spillage/output/spillage_dataset.pkl


## 4 · Inspect a Sample

In [ ]:
# Reload if re-running from here
if 'samples' not in dir():
    with open(DATASET_PATH, 'rb') as f:
        samples = pickle.load(f)

viz_utils.show_sample(samples[0], title=f"Arena={samples[0]['metadata']['arena_id']}  Path={samples[0]['metadata']['path_id']}  Step={samples[0]['metadata']['step_id']}")

## 5 · Train

In [ ]:
train_data, val_data = model_utils.split_samples(samples, val_ratio=0.20, seed=42)
print(f"Train: {len(train_data)}  Val: {len(val_data)}")

model, history, device = model_utils.train_model(
    train_data, val_data,
    epochs=25,
    batch_size=32,
    lr=1e-3,
)
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

Train: 115  Val: 29


epochs:   0%|          | 0/25 [00:00<?, ?it/s]

Final val loss: 0.0100


In [ ]:
viz_utils.show_loss(history)

## 6 · Evaluate & Visualise

In [ ]:
import torch

model.eval()
for i in range(min(3, len(val_data))):
    s = val_data[i]
    x_t = torch.from_numpy(s['input']).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x_t).squeeze(0).cpu().numpy()
    viz_utils.show_sample(
        s, pred=pred,
        title=f"[Val #{i}] Arena={s['metadata']['arena_id']}  Path={s['metadata']['path_id']}  Step={s['metadata']['step_id']}"
    )